# Word2Vec implementation

Below you can find my implementation of Word2Vec using skip-gram method with negative sampling. Therefore, the loss function I am maximizing is this

$$\log \sigma \left( {V'_{w_O}}^\top V_{w_I} \right) + \sum_{i=1}^{k} \mathbb{E}_{w_i \sim P_n(w)} \left[ \log \sigma \left( -{V'_{w_i}}^\top V_{w_I} \right) \right]$$

where $V$ is the projection matrix and $V'$ is the output matrix (I took this maximizing function from [this presentation](https://ufal.mff.cuni.cz/~zabokrtsky/courses/npfl124/slides/deep-learning-intro.pdf)). The embeddings that I choose are rows of the projection matrix.

In [ ]:
import numpy as np
from typing import Dict, Tuple, Union, List

def log_sigmoid(x):
    """Log sigmoid function."""
    return np.log(1.0 / (1.0 + np.exp(-x)))

def log_sigmoid_derivative(x):
    """Derivative of log sigmoid function."""
    return 1.0/(1.0 + np.exp(x))

class Word2Vec:
    """An implementation of the Word2Vec model using negative sampling.
    
    Attributes:
        vocab (Dict[str, int]): A mapping from words to their corresponding indices in the vocabulary
        index_to_word (Dict[int, str]): A mapping from indices to their corresponding words in the vocabulary
        embedding_dim (int): The dimensionality of the word embeddings
        vocab_size (int): The size of the vocabulary
        random_batch_factor (int): A factor to control the number of random negative samples drawn in each iteration
        projection_matrix (np.ndarray): A matrix of shape (vocab_size, embedding_dim) that contains the input word embeddings
        output_matrix (np.ndarray): A matrix of shape (embedding_dim, vocab_size) that contains the output word embeddings
    """
    def __init__(self, vocab, embedding_dim, seed, random_batch_factor = 5) -> None:
        self._rng = np.random.default_rng(seed)
        self.embedding_dim = embedding_dim
        self.vocab : Dict[str, int] = vocab
        self.index_to_word : Dict[int, str] = {v: k for k, v in vocab.items()}
        self.vocab_size = len(vocab)
        self.random_batch_factor = random_batch_factor
        self.projection_matrix = (self._rng.random((self.vocab_size, self.embedding_dim), dtype=np.float32) - 0.5) / self.embedding_dim
        self.output_matrix = (self._rng.random((self.embedding_dim, self.vocab_size), dtype=np.float32) - 0.5) / self.embedding_dim

    
    def gradient_asc(self, input_word_index : int, output_word_index : int, random_word_indexes : np.ndarray, learning_rate : float):
        """
        Performs a single step of gradient ascent for the given input word, output word, and negative samples.
        The method calculates the gradients for the input and output word embeddings based on the positive and negative samples and updates the projection and output matrices accordingly.
        """
        input_word_grad = np.zeros_like(self.projection_matrix[input_word_index])

        output_word_gradient_base = log_sigmoid_derivative(self.embedding_dot(input_word_index, output_word_index))
        output_word_gradient = self.projection_matrix[input_word_index] * output_word_gradient_base
        self.output_matrix[:, output_word_index] += learning_rate * output_word_gradient

        input_word_grad += self.output_matrix[:, output_word_index] * output_word_gradient_base

        for i in random_word_indexes:
            negative_example_gradient_base = log_sigmoid_derivative(-self.embedding_dot(input_word_index, i))
            negative_example_gradient = -self.projection_matrix[input_word_index] * negative_example_gradient_base
            self.output_matrix[:, i] += learning_rate * negative_example_gradient
            
            input_word_grad -= self.output_matrix[:, i] * negative_example_gradient_base
        
        self.projection_matrix[input_word_index] += learning_rate * input_word_grad

    def random_negative_samples(self, window : List[int], k: int = 5):
        """
        Draws k random negative samples from the vocabulary, ensuring that none of the samples are present in the provided window of context words. 
        The method uses a while loop to continuously draw random samples until it has collected k valid negative samples that are not in the window or already in the output list.
        """
        banned = set(window)
        out = []
        draw = self._rng.integers
        while len(out) < k:
            batch = draw(0, self.vocab_size, size=(k - len(out)) * self.random_batch_factor)
            for c in batch:
                if c not in banned and c not in out:
                    out.append(int(c))
                    if len(out) == k:
                        break
        return np.array(out, dtype=np.int64)

    def criterion(self, input_word_index : int, output_word_index : int, random_word_indexes : np.ndarray) -> Tuple[float, np.ndarray, np.ndarray]:
        """
        Computes the loss for a given input word, output word, and negative samples.
        """
        positive_score = log_sigmoid(self.embedding_dot(input_word_index, output_word_index))
        negative_scores = log_sigmoid(-self.embedding_dot(input_word_index, random_word_indexes))
        score = positive_score + np.sum(negative_scores)
        return score

    def embedding_dot(self, input_word_index : int, output_word_index : Union[int, np.ndarray]) -> float:
        """
        Computes the dot product between the input word embedding and the output word embedding(s).
        """
        input_vector = self.projection_matrix[input_word_index]
        output_vector = self.output_matrix[:, output_word_index]
        score = np.dot(input_vector, output_vector)
        return score

    def train(self, tokenized_corpus : List[int], window_size, learning_rate, epochs):
        """
        Trains the Word2Vec model using the provided tokenized corpus. 
        All tokens in the corpus should be represented as their corresponding indices in the vocabulary.
        """
        for epoch in range(epochs):
            for i in range(len(tokenized_corpus)):
                input_word = tokenized_corpus[i]
                window_start = max(0, i - window_size)
                window_end = min(len(tokenized_corpus), i + window_size + 1)
                window = tokenized_corpus[window_start:window_end]
                
                random_word_indexes = self.random_negative_samples(window)
                for output_word in window:
                    if output_word != input_word:
                        score = self.criterion(input_word, output_word, random_word_indexes)
                        self.gradient_asc(input_word, output_word, random_word_indexes, learning_rate)
    def get_embedding(self, word):
        """
        Retrieves the embedding vector for a given word. 
        It looks up the index of the word in the vocabulary and then returns the corresponding row from the projection matrix.
        """
        index = self.vocab[word]
        return self.projection_matrix[index]
    def get_nearest_neighbors(self, target_vector, k=10):
        """
        Finds the k nearest words to the target_vector using Cosine Similarity. 
        Normalizes both the target vector and the projection matrix and then calculates the dot product. 
        The dot products are then sorted to find the top k nearest neighbors.

        This is just a debugging method to see if the model is learning something.
        """
        target_norm = np.linalg.norm(target_vector)
        assert target_norm > 0, "Target vector must not be the zero vector."
        target_vector = target_vector / target_norm

        matrix_norms = np.linalg.norm(self.projection_matrix, axis=1, keepdims=True)
        matrix_norms[matrix_norms == 0] = 1e-10 
        normalized_matrix = self.projection_matrix / matrix_norms

        # Because x and y is normalized, x dot y = ||x|| * ||y|| * cos(theta) = cos(theta)
        similarities = np.dot(normalized_matrix, target_vector)
        top_k_indices = np.argsort(similarities)[-k:][::-1]
        
        return [(self.index_to_word[idx], similarities[idx]) for idx in top_k_indices]

## Helper function

In [7]:
def debug_prints(word: str):
    print(f"  nearest to '{word}': ")
    nearest = model.get_nearest_neighbors(model.get_embedding(word), k=5)
    for word, proximity in nearest:
        print(f"    {word}: {proximity}")

## Example

Here I am training on a slimmed-down movie_reviews dataset from nltk corpora because I don't have enough compute power to test the algorithm on larger datasets. Below I retrieve the dataset from the internet and then choose a subset of the reviews to train the model on.

### Dataset download

In [3]:
import requests
import zipfile

response = requests.get('https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/movie_reviews.zip')
with open('movie_reviews.zip', 'wb') as f:
    f.write(response.content)
with zipfile.ZipFile('movie_reviews.zip', 'r') as z:
    z.extractall()

### Preprocessing the dataset

In [4]:
import re
import os
shuffle_rng = np.random.default_rng(42)

negative_files = os.listdir('movie_reviews/neg')
shuffle_rng.shuffle(negative_files)
negative_reviews = []
for i in range(300):
    with open(f'movie_reviews/neg/{negative_files[i]}', 'r') as f:
        text = f.read().lower()
        text = re.sub("[^a-z' ]", "", text)
    negative_reviews.append(text.split())
positive_files = os.listdir('movie_reviews/pos')
shuffle_rng.shuffle(positive_files)
positive_reviews = []
for i in range(300):
    with open(f'movie_reviews/pos/{positive_files[i]}', 'r') as f:
        text = f.read()
        text = re.sub(r"[^a-z' ]", "", text)
    positive_reviews.append(text.split())
tokens = [word for review in negative_reviews + positive_reviews for word in review]
vocab = {word: idx for idx, word in enumerate(set(tokens))}

### Model initialization

In [5]:
embedding_dim = 50
seed = 42
model = Word2Vec(vocab, embedding_dim, seed=seed)

### Training

In [ ]:
debug_prints('good')
for i in range(3):
    print(f"Epoch {i}")
    for review in negative_reviews + positive_reviews:
        tokenized_review = [vocab[word] for word in review]
        model.train(tokenized_review, window_size=10, learning_rate=0.005, epochs=1)
    debug_prints('good')

  nearest to 'good': 
    good: 0.9999998807907104
    dud: 0.5681523084640503
    mortgages: 0.5060724020004272
    sweden: 0.4969484210014343
    ermey's: 0.47899842262268066
Epoch 0
  nearest to 'good': 
    good: 1.0
    special: 0.9994366765022278
    classic: 0.9993911385536194
    waste: 0.9993559122085571
    however: 0.9993271827697754
Epoch 1
  nearest to 'good': 
    good: 1.0000001192092896
    between: 0.9936361908912659
    man: 0.9929112792015076
    great: 0.9883047938346863
    character: 0.9880651235580444
Epoch 2


/tmp/ipykernel_186471/672031617.py:10: RuntimeWarning: overflow encountered in exp
  return 1.0/(1.0 + np.exp(x))
/tmp/ipykernel_186471/672031617.py:6: RuntimeWarning: overflow encountered in exp
  return np.log(1.0 / (1.0 + np.exp(-x)))
/tmp/ipykernel_186471/672031617.py:6: RuntimeWarning: divide by zero encountered in log
  return np.log(1.0 / (1.0 + np.exp(-x)))


  nearest to 'good': 
    good: 0.9999999403953552
    bit: 0.9745758175849915
    comedy: 0.9729340076446533
    lot: 0.9711104035377502
    great: 0.9681919813156128
Epoch 3
  nearest to 'good': 
    good: 0.9999999403953552
    great: 0.9236446619033813
    sense: 0.9216688871383667
    nice: 0.9184783697128296
    just: 0.9179245233535767
Epoch 4
  nearest to 'good': 
    good: 1.0000001192092896
    nice: 0.8801941275596619
    very: 0.8795171976089478
    few: 0.8772171139717102
    lot: 0.8769521713256836
